# 2. Clustering i unsupervised

DBSCAN, HDBSCAN, K-Means, lokalne embeddingi LSA, Isolation Forest i demonstracja wykrywania nowej kampanii.

## Protokół wspólny

Dane lokalne: pełny `combined_reduced.json` (77 677 rekordów, `1` = phishing,
`0` = benign); mieszanina URL, SMS, e-maili i HTML, a nie tylko e-maile.
Każdy notatnik sprawdza SHA-256 wszystkich części i oryginału.
Po odrzuceniu pustych wejść, konfliktów etykiet i duplikatów znormalizowanych
pierwszych 4000 znaków wybieramy warstwowo **8000 rekordów** z ziarnem 42.
Używamy tego samego podziału train/validation/test = **60/20/20**.
To demonstracja na próbce; pełny zbiór pozostaje w `data`.

Wszystkie reprezentacje uczymy tylko na train. Walidacja służy do wyboru progu
w metodach nadzorowanych; test zostaje na końcowy pomiar. Zbiory są rozłączne
po kluczu normalizacji, lecz bliskie duplikaty lub rodziny kampanii nadal mogą
przeciekać: brak dat i identyfikatorów kampanii uniemożliwia wiarygodny test czasowy.
Nie odwiedzamy linków ani nie renderujemy HTML. Nie korzystamy z API modeli,
internetu podczas wykonania ani pobieranych modeli językowych.

Czasy to pomiar CPU w tej maszynie, z mediany 3 wywołań po rozgrzewce, w ms/próbkę
w batchu testowym (nie opóźnienie pojedynczego żądania). Obejmują transformację tekstu,
ale nie odczyt plików. Rozmiar to nieskompresowana serializacja pickle obiektu
potrzebnego do predykcji, bez środowiska Pythona i danych. Wartości między rodzinami
zadań nie są automatycznie porównywalne. Czułość = recall klasy phishing;
FPR = FP / (FP + TN). Average precision podsumowuje ranking precision–recall.

In [1]:
import sys
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Uruchomienie z katalogu repozytorium albo z notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'notebooks' / 'common.py').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from common import (SEED, load_splits, classification_metrics, choose_threshold,
                    timed_prediction, size_mib, save_results)
train, validation, test = load_splits()
y_train = train.label.to_numpy()
y_val = validation.label.to_numpy()
y_test = test.label.to_numpy()
x_train = train.input.tolist()
x_val = validation.input.tolist()
x_test = test.input.tolist()
display(pd.DataFrame({'train': train.label.value_counts(),
                      'validation': validation.label.value_counts(),
                      'test': test.label.value_counts()}).rename_axis('label'))

{
  "source_records": 77677,
  "deduplicated_sample": 8000,
  "conflicting_keys_removed": 3,
  "seed": 42,
  "train": 4800,
  "validation": 1600,
  "test": 1600,
  "max_input_chars": 4000,
  "split_sha256": {
    "train": "dbd41bff6349f412f1cbc777e2a7e19e75f23d09586914fd0518a620cd0caf18",
    "validation": "5ef4b8c140c03e6f97c2a792e2887a785aac5d59986fd08ea41784e87d85904b",
    "test": "ce29c3a3e537a8f6d97ad38fd43fe2c5952ff6ed04c7611991a05c8784e0aa5c"
  }
}


,train,validation,test
label,,,
0,2802,934,934
1,1998,666,666


## Reprezentacja bez etykiet

TF-IDF word 1–2-gram (max 12 000 cech) dopasowujemy na train.
Tworzymy też **64-wymiarowe embeddingi LSA** przez TruncatedSVD i normalizację L2.
Są to lokalne wektory uczone z tego datasetu, nie embeddingi pretrenowanego BERT.
SVD kompresuje współwystępowanie słów i nie wymaga modeli ani API zewnętrznych.
Etykiety nie są używane do uczenia reprezentacji, wyboru eps, k ani dopasowania klastrów.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import DBSCAN, HDBSCAN, KMeans
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (adjusted_rand_score, normalized_mutual_info_score,
                             silhouette_score)
start = perf_counter()
vectorizer = TfidfVectorizer(max_features=12000, ngram_range=(1, 2), min_df=2,
                             sublinear_tf=True, dtype=np.float32)
T = vectorizer.fit_transform(x_train)
tfidf_only_s = perf_counter() - start
svd = TruncatedSVD(n_components=64, random_state=SEED)
E = normalize(svd.fit_transform(T))
representation_s = perf_counter() - start
def embed(texts):
    return normalize(svd.transform(vectorizer.transform(texts)))
print('TF-IDF:', T.shape, 'LSA:', E.shape,
      'explained variance:', round(float(svd.explained_variance_ratio_.sum()), 4))
# eps z rozkładu odległości 8. sąsiada, bez etykiet i bez testu.
neighbors = NearestNeighbors(n_neighbors=9, n_jobs=1).fit(E)
distances, _ = neighbors.kneighbors(E)
eps = float(np.quantile(distances[:, -1], 0.65))
eps = max(eps, 1e-4)
print('DBSCAN eps:', eps)

TF-IDF: (4800, 12000) LSA: (4800, 64) explained variance: 0.3497
DBSCAN eps: 0.4662992030382156


## Cztery warianty klasteryzacji

DBSCAN: eps z train, min_samples=8. HDBSCAN: min_cluster_size=15, min_samples=8,
zmienna gęstość. K-Means: k=12, n_init=10, zarówno w TF-IDF jak i LSA.
Liczby klastrów nie utożsamiamy z dwiema klasami phishing/benign.

ARI i NMI porównują klastry z etykietami **dopiero po dopasowaniu** na train.
Silhouette liczymy na maks. 1500 niezaszumionych punktów, jeśli istnieją ≥2 klastry.
Szum ma etykietę `-1`; ARI/NMI obejmują również szum. Te metryki nie są F1 filtra.

In [3]:
cluster_specs = [
    ('DBSCAN / LSA', DBSCAN(eps=eps, min_samples=8, n_jobs=1), E, 'LSA'),
    ('HDBSCAN / LSA', HDBSCAN(min_cluster_size=15, min_samples=8, n_jobs=1, copy=True), E, 'LSA'),
    ('K-Means / TF-IDF', KMeans(n_clusters=12, n_init=10, random_state=SEED), T, 'TF-IDF'),
    ('K-Means / LSA', KMeans(n_clusters=12, n_init=10, random_state=SEED), E, 'LSA'),
]
rows, fitted = [], {}
for name, model, X, space in cluster_specs:
    start = perf_counter()
    labels = model.fit_predict(X)
    fit_s = perf_counter() - start
    mask = labels != -1
    unique = np.unique(labels[mask])
    silhouette = np.nan
    if 1 < len(unique) < mask.sum():
        silhouette = silhouette_score(X[mask], labels[mask], sample_size=min(1500, mask.sum()),
                                      random_state=SEED)
    fitted[name] = (model, labels)
    rows.append(dict(method=name, task='clustering', clusters=len(unique), noise=float((~mask).mean()),
                     silhouette=silhouette, ari=adjusted_rand_score(y_train, labels),
                     nmi=normalized_mutual_info_score(y_train, labels),
                     fit_s=fit_s + representation_s,
                     model_mib=size_mib((vectorizer, svd if space == 'LSA' else None, model))))
display(pd.DataFrame(rows).round(4))
# Profil klastrów: same agregaty, bez wyświetlania URL/HTML z próbek.
km, km_labels = fitted['K-Means / LSA']
display(pd.DataFrame({'cluster': km_labels, 'phishing': y_train})
        .groupby('cluster').phishing.agg(['size', 'mean']).rename(columns={'mean':'phishing_share'}))

,method,task,clusters,noise,silhouette,ari,nmi,fit_s,model_mib
0,DBSCAN / LSA,clustering,65,0.2412,0.4421,0.0081,0.0760,1.6445,4.1779
1,HDBSCAN / LSA,clustering,52,0.2315,0.4753,0.0088,0.0732,3.0737,5.8932
2,K-Means / TF-IDF,clustering,12,0.0000,0.0803,0.0158,0.0391,4.2646,0.9662
3,K-Means / LSA,clustering,12,0.0000,0.1836,0.0357,0.0636,1.6870,3.3507


,size,phishing_share
cluster,,
0,178,0.505618
1,200,0.200000
2,428,0.801402
3,416,0.199519
4,593,0.347386
5,237,0.582278
6,731,0.160055
7,375,0.794667
8,202,0.287129


## Obsługa nowych wiadomości i pomiar czasu

K-Means ma natywne `predict`. DBSCAN nie ma go: przydzielamy nową próbkę
do najbliższego punktu rdzeniowego tylko w promieniu eps; inaczej jest szumem.
To jawna heurystyka wdrożeniowa, nie ponowny DBSCAN na strumieniu.
Implementacja sklearn HDBSCAN nie ma `predict`; oceniamy go jako metodę batchową,
więc czas dotyczy ponownego fit_predict na batchu testowym. Nie porównujemy tej
wartości bezpośrednio z czasem predykcji online innych metod.

In [4]:
for row in rows:
    model, _ = fitted[row['method']]
    if isinstance(model, KMeans):
        space = 'TF-IDF' if row['method'].endswith('TF-IDF') else 'LSA'
        def predict(texts, m=model, s=space):
            return m.predict(vectorizer.transform(texts) if s == 'TF-IDF' else embed(texts))
        _, latency = timed_prediction(predict, x_test)
        row['timing_mode'] = 'transform + predict'
    elif isinstance(model, DBSCAN):
        cores = model.core_sample_indices_
        if len(cores):
            core_nn = NearestNeighbors(n_neighbors=1, n_jobs=1).fit(E[cores])
        def predict(texts, m=model, cores=cores):
            Z = embed(texts)
            if not len(cores):
                return np.full(len(texts), -1)
            distance, index = core_nn.kneighbors(Z)
            return np.where(distance[:, 0] <= m.eps, m.labels_[cores[index[:, 0]]], -1)
        predicted, latency = timed_prediction(predict, x_test)
        row['test_noise'] = float(np.mean(predicted == -1))
        row['model_mib'] = size_mib((vectorizer, svd, model,
                                     core_nn if len(cores) else None))
        row['timing_mode'] = 'transform + nearest core (heurystyka)'
    else:
        # Nowy obiekt przy każdej próbie; nie nadpisujemy modelu wyuczonego na train.
        def batch(texts):
            return HDBSCAN(min_cluster_size=15, min_samples=8, n_jobs=1, copy=True).fit_predict(embed(texts))
        _, latency = timed_prediction(batch, x_test)
        row['timing_mode'] = 'transform + batch fit_predict (nie online)'
    row['ms_per_sample'] = latency

## Isolation Forest (IsoForest): anomalia ≠ phishing

Uczymy na **całym train bez etykiet**, contamination=0.10. Model zgłasza ok. 10%
treningowych obserwacji jako anomalie, chociaż udział phishingu jest inny.
Wynik to `−decision_function`, więc większa wartość = większa anomalia;
próg 0 jest ustalony przez model, bez dostrajania na etykietach.
Dopiero na teście porównujemy anomalie do etykiet phishing. Niska czułość
nie jest błędem instalacji: częste kampanie mogą nie być odstające.

In [5]:
start = perf_counter()
iso = IsolationForest(n_estimators=150, contamination=0.10, random_state=SEED, n_jobs=1)
iso.fit(E)
iso_fit_s = perf_counter() - start
def anomaly_scores(texts):
    return -iso.decision_function(embed(texts))
scores, latency = timed_prediction(anomaly_scores, x_test)
iso_row = dict(method='Isolation Forest / LSA', task='anomaly detection',
               fit_s=representation_s + iso_fit_s, ms_per_sample=latency,
               timing_mode='transform + decision_function', model_mib=size_mib((vectorizer, svd, iso)),
               **classification_metrics(y_test, scores >= 0, scores))
rows.append(iso_row)
display(pd.DataFrame([iso_row]).round(4))

,method,task,fit_s,ms_per_sample,timing_mode,model_mib,accuracy,precision,recall,f1,fpr,roc_auc,average_precision,tn,fp,fn,tp
0,Isolation Forest / LSA,anomaly detection,1.7297,0.1197,transform + decision_function,4.4731,0.62,0.6883,0.1592,0.2585,0.0514,0.5762,0.5422,886,48,560,106


## Nowa kampania: kontrolowana symulacja

Dataset nie zawiera czasu ani identyfikatorów kampanii, więc nie można udowodnić
odporności na rzeczywisty Zero-Day. Symulujemy **40 wariantów nowej kampanii**:
bierzemy prefiksy 40 testowych wiadomości phishing i dodajemy wspólny tekst
nakłaniający do weryfikacji konta. Ten sygnał nie był dodawany do train.
Nie wykonujemy adresów ani nie generujemy aktywnych stron.

Reprezentacja zostaje zamrożona na train. DBSCAN dopasowujemy do train + nowego
batcha bez etykiet. Raportujemy skupienie wariantów i ich separację od train,
zamiast ogłaszać, że każdy nowy klaster jest phishingiem. Użycie etykiet do
skonstruowania symulacji nie jest częścią detektora. Taki test bywa łatwiejszy
niż prawdziwa, zróżnicowana kampania; nie zastępuje testu czasowego.

In [6]:
campaign_source = test.loc[test.label == 1, 'input'].head(40).tolist()
assert len(campaign_source) == 40
shared = (' urgent account verification confirm password secure login update account ' * 12)
campaign = [shared + text[:100] + f' reference {i}' for i, text in enumerate(campaign_source)]
campaign_E = embed(campaign)
batch_E = np.vstack([E, campaign_E])
campaign_labels = DBSCAN(eps=eps, min_samples=8, n_jobs=1).fit_predict(batch_E)
baseline_labels, incoming_labels = campaign_labels[:-40], campaign_labels[-40:]
groups = []
for label in np.unique(incoming_labels):
    groups.append(dict(cluster=int(label), simulated_messages=int((incoming_labels == label).sum()),
                       train_messages=int((baseline_labels == label).sum()),
                       new_dense_group=bool(label != -1 and not (baseline_labels == label).any())))
campaign_summary = pd.DataFrame(groups)
display(campaign_summary)
(ROOT / 'results').mkdir(exist_ok=True)
campaign_summary.to_csv(ROOT / 'results' / '02_symulacja_kampanii.csv', index=False)
print('Udział wariantów poza szumem:', float((incoming_labels != -1).mean()))

,cluster,simulated_messages,train_messages,new_dense_group
0,-1,2,1156,False
1,39,38,14,False


Udział wariantów poza szumem: 0.95


## Łączne podsumowanie porównawcze

Tabela obejmuje DBSCAN, HDBSCAN, oba K-Means i Isolation Forest.
Puste pola są celowe: klasyfikacyjne precision/recall/F1 dotyczą wyłącznie
Isolation Forest, a ARI/NMI/silhouette dotyczą klastrów. Nie mapujemy arbitralnie
numeru klastra na etykietę phishing. `fit_s` zawiera wspólny koszt TF-IDF + SVD
(dla K-Means TF-IDF poniżej odejmujemy zbędny koszt SVD na podstawie pomiaru etapu TF-IDF).

In [7]:
# Dla wariantu bez SVD uwzględnij tylko rzeczywiście zmierzony fit TF-IDF.
for row in rows:
    if row['method'] == 'K-Means / TF-IDF':
        row['fit_s'] += tfidf_only_s - representation_s
summary = save_results('02_clustering_i_unsupervised', rows)
display(summary.round(4))
assert len(summary) == 5
display(Markdown(f"Isolation Forest: czułość **{iso_row['recall']:.3f}**, "
                 f"FPR **{iso_row['fpr']:.3f}**. "
                 f"DBSCAN eps dobrane bez etykiet: **{eps:.3f}**. "
                 "Wynik symulacji kampanii jest pokazany osobno powyżej."))

,method,task,clusters,noise,silhouette,ari,nmi,fit_s,model_mib,test_noise,...,precision,recall,f1,fpr,roc_auc,average_precision,tn,fp,fn,tp
0,DBSCAN / LSA,clustering,65.0,0.2412,0.4421,0.0081,0.0760,1.6445,4.9656,0.2556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,HDBSCAN / LSA,clustering,52.0,0.2315,0.4753,0.0088,0.0732,3.0737,5.8932,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,K-Means / TF-IDF,clustering,12.0,0.0000,0.0803,0.0158,0.0391,4.0971,0.9662,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,K-Means / LSA,clustering,12.0,0.0000,0.1836,0.0357,0.0636,1.6870,3.3507,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Isolation Forest / LSA,anomaly detection,NaN,NaN,NaN,NaN,NaN,1.7297,4.4731,NaN,...,0.6883,0.1592,0.2585,0.0514,0.5762,0.5422,886.0,48.0,560.0,106.0


Isolation Forest: czułość **0.159**, FPR **0.051**. DBSCAN eps dobrane bez etykiet: **0.466**. Wynik symulacji kampanii jest pokazany osobno powyżej.

**Wnioski:** DBSCAN rozpoznaje gęste rodziny i szum, lecz wymaga sensownego eps.
HDBSCAN obsługuje zmienną gęstość i nie wymaga k, ale może odrzucać znaczną część
próbek. K-Means jest praktyczny dla dużych batchy, wymusza jednak przydział każdej
próbki i kuliste grupy. LSA kompresuje rzadkie wektory i może wygładzać szczegóły
URL, dlatego porównujemy je z TF-IDF. Isolation Forest priorytetyzuje nietypowe
wiadomości, ale phishing nie musi być nietypowy. Nowa gęsta grupa jest sygnałem
dla analityka, nie dowodem oszustwa. Ceny pamięci i opóźnień zależą od wymiaru,
liczby wiadomości oraz trybu batch/online. Parametrów nie optymalizowano etykietami.

W zapisanym eksperymencie żaden klaster symulowanych wiadomości nie spełnił
warunku „nowa gęsta grupa bez punktów train”. Warianty dołączyły do grup obecnych
w treningu. To negatywny wynik tej konkretnej heurystyki; zamrożone TF-IDF/LSA
oraz próg gęstości nie zapewniły separacji nowej kampanii. Wysokie skupienie
wariantów w grupie nie jest równoznaczne z wykryciem jej nowości.